# 09 — The QOT Zoo

Put all three quantum optimal-transport objects on one bench and ask the question the module has been circling: when do they give the *same* number, and when do they part ways? The honest answer is the whole point — they are a **reconciled family**, agreeing exactly in some regimes, ordered identically in others, and genuinely distinct in the rest.

**Prerequisites:** `08_quantum_amari_bridge`, and `03/13_the_bures_bridge`.

**What you'll be able to do**
- Name the **three QOT objects** the course built — the Bures bridge, the coupling SDP, and the entropic object — and say which notebook each came from.
- Watch the **entropic object converge to the coupling SDP** as the regularisation $\varepsilon \to 0$, the convergence pinned by a regression test in the suite.
- Confirm the **diagonal-collapse common limit**: on commuting states with the quadratic-position cost, the SDP returns the classical $W_2^2$ — the shared classical floor all three sit above.
- State the **Bures-versus-SDP relationship precisely**: on pure states with the SWAP cost both are monotone functions of the *same* fidelity, so they are *order-equivalent* but **not equal** — and read a regimes table that lays out exactly where the three coincide, where they only agree on ordering, and where they differ.

You have built three different doors into quantum optimal transport, and today you find out how the rooms behind them connect. This is a coherence anchor, not a new construction: every object here is one you already made. What is new is the map between them — and the discipline to state that map exactly. The temptation is to declare the three "the same"; the truth is more interesting, and getting it right is the work of this notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ot

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.sdp import (
    quadratic_position_cost,
    quantum_ot_sdp,
    quantum_wasserstein_squared_swap,
)
from qot_course.quantum_ot.sinkhorn import quantum_sinkhorn_cost
from qot_course.infotheory.quantum import bures_distance
from qot_course.transport.gaussian import bures_matrix_distance
from qot_course.quantum.density import density_matrix, fidelity
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS, qubit_state

np.random.seed(42)
viz.use_course_style()

## 1. Three objects, one subject

Across Modules 3 and 4 you built quantum optimal transport three times over, from three different starting points. Here they are, one line each, with the notebook that built it:

- **The Bures bridge** — `03/13`. You fed the Gaussian Wasserstein matrix term *density matrices* instead of covariances and it became the squared quantum **Bures distance** $d_B^2(\rho, \sigma) = 2\,(1 - F_U)$, where $F_U$ is the Uhlmann fidelity. This was the doorway: a classical transport formula reading a quantum state.
- **The coupling SDP** — `04/04`. You lifted the Kantorovich LP to operators and solved $\min_{\rho_{AB}\succeq 0}\ \mathrm{tr}(C\,\rho_{AB})$ over quantum couplings $\rho_{AB}$ with partial traces $\rho_A, \rho_B$. The PSD cone replaced the non-negative orthant; the cost operator $C$ encodes what "far apart" means.
- **The entropic object** — `04/06`–`04/08`. You added a von Neumann entropy bonus, $\min\ \mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})$, and `04/08` revealed its geometric identity: the entropic plan is the **Umegaki projection** of the matrix-exponential Gibbs kernel $K = \exp(-C/\varepsilon)$ onto the coupling set.

Three objects, one subject. The reconciliation has three parts, and they are *not* the same kind of statement — so we take them one at a time and say exactly what each one claims. Two of the three live naturally with the **quadratic-position cost** on commuting states (the entropic and diagonal-collapse parts); the Bures relationship is cleanest on **pure states with the SWAP cost**. We will use each object where it is on solid numerical ground.

## 2. The entropic object converges to the coupling SDP as $\varepsilon \to 0$

The first reconciliation is the cleanest, and it is exact in the limit. The entropic objective is the coupling-SDP objective plus an entropy bonus weighted by $\varepsilon$:

$$ \underbrace{\mathrm{tr}(C\,\rho_{AB}) - \varepsilon\,S(\rho_{AB})}_{\text{entropic (04/06)}} \quad\xrightarrow{\ \varepsilon \to 0\ }\quad \underbrace{\mathrm{tr}(C\,\rho_{AB})}_{\text{coupling SDP (04/04)}}. $$

As $\varepsilon$ shrinks, the entropy term fades and the entropic optimum slides toward the unregularised SDP optimum. So the *transport cost* read off the entropic plan, $\mathrm{tr}(C\,\rho^\star_\varepsilon)$, should approach the SDP value. This is exactly what the regression test `test_entropic_cost_converges_to_sdp_as_epsilon_to_zero` pins in the suite — and we make it *visible* here by sweeping $\varepsilon$ from large down to small and watching the gap collapse.

We use the **commuting diagonal pair** the entropic arc has used throughout: the quadratic-position cost on positions $\{0, 1, 2\}$ with $\rho_A = \mathrm{diag}(0.5, 0.3, 0.2)$ and $\rho_B = \mathrm{diag}(0.2, 0.3, 0.5)$. (We avoid the headline $|+\rangle\langle+|$ versus $I/2$ pair for entropic solves: its joint entropy is pinned by the Araki–Lieb bound, which puts the entropic optimum on the PSD-cone boundary and sends the solver to infinity. The commuting pair keeps every entropic plan strictly positive definite.)

In [ ]:
# The full-rank commuting pair from the entropic arc (04/06-08).
positions = np.array([0.0, 1.0, 2.0])
C = quadratic_position_cost(positions)
rho_a = np.diag([0.5, 0.3, 0.2]).astype(complex)
rho_b = np.diag([0.2, 0.3, 0.5]).astype(complex)

# The unregularised coupling-SDP value -- the target the entropic cost approaches.
sdp_value, _ = quantum_ot_sdp(rho_a, rho_b, C)

# Sweep epsilon large -> small and read the transport cost tr(C P_eps) off each
# entropic plan. (A CLARABEL "solution may be inaccurate" warning may print as eps
# gets small -- it is the interior-point method reporting it stopped near its
# tightened tolerance. We keep it: the value is the one we want.)
epsilons = [2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02]
entropic_costs = [quantum_sinkhorn_cost(rho_a, rho_b, C, epsilon=eps) for eps in epsilons]
gaps = [abs(c - sdp_value) for c in entropic_costs]

print(f"coupling-SDP value  tr(C P*)  = {sdp_value:.6f}   (the limit)")
print()
print(f"{'epsilon':>9}   {'entropic cost tr(C P_eps)':>26}   {'gap to SDP':>12}")
print("-" * 53)
for eps, cost, gap in zip(epsilons, entropic_costs, gaps):
    print(f"{eps:>9.2f}   {cost:>26.6f}   {gap:>12.2e}")

**Read the output.** Run your eye down the table from top to bottom. At $\varepsilon = 2$ the entropy bonus is strong: the entropic plan is smeared out, its transport cost sits well above the SDP value, and the gap is large. As $\varepsilon$ falls the entropy term loosens its grip, the cost descends toward the SDP value, and the **gap column collapses** — from a sizeable number at $\varepsilon = 2$ to a residual at the solver-tolerance floor by the time $\varepsilon$ reaches the bottom of the sweep. That is the convergence the regression test guards, made visible: the entropic object is not a *different* transport cost, it is the coupling SDP seen through an entropy-smoothed lens, and as the lens clears the two coincide. The plot makes the collapse unmistakable.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(epsilons, gaps, marker="o", color=COLORS["quantum"], linewidth=2.0,
          markersize=8, label=r"gap  $|\,\mathrm{tr}(C\,P_\varepsilon) - \mathrm{SDP}\,|$")
ax.set_xlabel(r"regularisation strength  $\varepsilon$  (log scale)")
ax.set_ylabel("gap to the coupling-SDP value (log scale)")
ax.set_title(r"Entropic $\to$ coupling SDP: the gap collapses as $\varepsilon \to 0$", pad=12)
ax.invert_xaxis()  # read left-to-right as epsilon shrinks
ax.legend()
plt.show()

**Read the figure.** The horizontal axis is $\varepsilon$ on a log scale, **inverted** so that reading left to right walks $\varepsilon$ from large down toward zero — the direction of the limit. The vertical axis is the gap between the entropic transport cost and the coupling-SDP value, also logarithmic. The curve falls steeply and steadily: each tenfold reduction in $\varepsilon$ buys roughly two orders of magnitude off the gap, until it flattens onto the solver's tolerance floor — the point past which the residual is numerical noise, not a real disagreement. A straight, plunging line on a log–log plot is the signature of genuine convergence. The entropic object and the coupling SDP meet at $\varepsilon = 0$; everything above is the smooth, regularised approach to that meeting.

## 3. The diagonal-collapse common limit — the shared classical floor

The second reconciliation reaches back to the consistency principle of `04/03` (the diagonal-collapse you proved in `04/01`). On **commuting** states — diagonal $\rho_A, \rho_B$ in the eigenbasis of the position observable — the quadratic-position cost operator $C = (X_A \otimes I - I \otimes X_B)^2$ reduces, term by term, to the classical squared-distance cost $C_{ij} = (i - j)^2$. The coherence the PSD cone could carry has nowhere to live on diagonal states, so the operator SDP collapses to the classical Kantorovich LP, and its value is the classical $W_2^2$:

$$ \mathrm{QOT}_{\text{quadratic}}(\rho_A, \rho_B) \;=\; W_2^2\big(\mathrm{diag}(\rho_A),\, \mathrm{diag}(\rho_B)\big) \qquad \text{(commuting case).} $$

This is the floor all three objects share. The entropic object sits above it (by the entropy term of §2, vanishing as $\varepsilon \to 0$), and on commuting states the SDP *is* it. We check the SDP against an independent classical solver — `ot.emd2` from POT, the exact linear-program solver — on the very same diagonal pair from §2, so the two sections share a state and the numbers line up.

In [ ]:
# Same commuting diagonal pair as section 2 -- so the two reconciliations share a state.
p = np.array([0.5, 0.3, 0.2])  # the diagonal of rho_a
q = np.array([0.2, 0.3, 0.5])  # the diagonal of rho_b

# Quantum side: the coupling SDP with the quadratic-position cost (this is sdp_value from
# section 2 -- re-solved here so the cell stands on its own).
qot_value, _ = quantum_ot_sdp(np.diag(p).astype(complex), np.diag(q).astype(complex), C)

# Classical side: the exact W_2^2 on the diagonals via POT's linear-program solver.
cost_classical = (positions.reshape(-1, 1) - positions.reshape(1, -1)) ** 2
classical_w2_sq = float(ot.emd2(p, q, cost_classical))

print(f"quantum coupling SDP   tr(C P*)            = {qot_value:.6f}")
print(f"classical W_2^2        ot.emd2 on diagonals = {classical_w2_sq:.6f}")
print(f"gap                                         = {abs(qot_value - classical_w2_sq):.2e}")
print(f"equal to solver tolerance?                    {bool(np.isclose(qot_value, classical_w2_sq, atol=1e-5))}")

**Read the output.** The quantum coupling SDP and the classical $W_2^2$ print the **same number** — their difference sits at the solver-tolerance floor, around $10^{-9}$. On commuting states the operator problem has collapsed onto the classical linear program: no coherence to exploit, so the PSD cone offers nothing the orthant did not, and the optimum is the classical optimal-transport cost exactly. This is the shared floor made concrete. All three objects descend to this same classical value in the commuting world — the SDP equals it here, and the entropic object (§2) limits to the SDP and therefore to it as well. Quantum optimal transport contains classical optimal transport as its diagonal special case, and that containment is what makes the whole lift trustworthy.

## 4. Bures and the SDP — *via the fidelity*, not by identity

The third reconciliation is the subtle one, and the one most easily overclaimed. It is tempting to say "the Bures bridge and the coupling SDP are the same object." They are **not**. They are two *different* monotone functions of the *same* underlying quantity — the fidelity — which makes them **order-equivalent** but **not equal**. Stating this precisely is the scientific heart of this notebook.

Work on **pure** states $\rho = |\psi\rangle\langle\psi|$ and $\sigma = |\phi\rangle\langle\phi|$, where both objects are clean. Write $F = |\langle\psi|\phi\rangle|^2$ for the fidelity (the Uhlmann fidelity *squared*, in this package's convention; on pure states it is the squared overlap). Then:

- the **coupling SDP with the SWAP cost** returns, by the closed form of `04/04`, $\ \mathrm{QOT}_{\mathrm{SWAP}} = 1 - F\,$;
- the **squared Bures distance** of `03/13` is $\ d_B^2 = 2\,(1 - \sqrt{F})\,$.

Both are strictly decreasing in $F$ — identical states ($F = 1$) give zero, orthogonal states ($F = 0$) give the maximum — so they rank every pair of states in the *same order*. But $1 - F$ and $2(1 - \sqrt{F})$ are different functions of $F$: they agree only at $F = 1$ (both zero — identical states), and differ everywhere else, most starkly at $F = 0$ where the SDP gives $1$ but the squared Bures distance gives $2$. Order-equivalent, not equal. We tabulate all three columns for several pure pairs and read the relationship off directly.

In [ ]:
pure_pairs = [
    ("|0> vs |1>",      density_matrix(KET_0),    density_matrix(KET_1)),
    ("|0> vs theta=pi/3", density_matrix(KET_0),  density_matrix(qubit_state(np.pi / 3))),
    ("|0> vs |+>",      density_matrix(KET_0),    density_matrix(KET_PLUS)),
    ("|0> vs theta=pi/6", density_matrix(KET_0),  density_matrix(qubit_state(np.pi / 6))),
]

print(f"{'pure pair':<20}{'F':>8}{'1 - F  (SDP-SWAP)':>20}{'SDP-SWAP value':>16}"
      f"{'2(1 - sqrt F)  (Bures^2)':>26}")
print("-" * 90)
swap_col, bures_col = [], []
for label, rho, sigma in pure_pairs:
    F = fidelity(rho, sigma)                       # |<psi|phi>|^2 on pure states
    one_minus_F = 1.0 - F                          # the SWAP-cost closed form
    sdp_swap = quantum_wasserstein_squared_swap(rho, sigma)  # the SDP, solved
    bures_sq = 2.0 * (1.0 - np.sqrt(F))            # the squared Bures distance
    swap_col.append(sdp_swap)
    bures_col.append(bures_sq)
    print(f"{label:<20}{F:>8.4f}{one_minus_F:>20.4f}{sdp_swap:>16.4f}{bures_sq:>26.4f}")

# Cross-check the squared Bures distance against the two src routines that compute it.
print()
for label, rho, sigma in pure_pairs:
    d_b_src = bures_distance(rho, sigma)                  # sqrt(2(1 - sqrt F))
    bw_matrix = bures_matrix_distance(rho, sigma)         # the BW matrix term = d_B^2
    print(f"{label:<20} d_B (03/13 route) = {d_b_src:.4f}   d_B^2 via BW matrix = {bw_matrix:.4f}")

**Read the output.** Three things to see, in order. **First**, the `1 - F` column and the **SDP-SWAP value** column match to four decimals on every row — the solver confirms the closed form, so the coupling SDP with the SWAP cost *is* $1 - F$. **Second**, the rightmost column, $2(1 - \sqrt{F})$, agrees with the SDP-SWAP column only at $F = 1$ (identical states, both $0$); at $F = 0$ (orthogonal $|0\rangle$ vs $|1\rangle$) the SDP-SWAP gives $1.0$ while the squared Bures distance gives $2.0$ — they do not agree there. In between — at $F = 0.5$, for instance — $1 - F = 0.5$ while $2(1 - \sqrt{F}) \approx 0.586$: the two are genuinely different numbers. **Third**, the cross-check block confirms the squared Bures distance equals $2(1 - \sqrt{F})$ through both `03/13` routes: `bures_distance` returns the distance $d_B = \sqrt{2(1 - \sqrt{F})}$ (we square it for the comparison), while `bures_matrix_distance` returns $d_B^2 = 2(1 - \sqrt{F})$ directly — both agree.

The reading is precise: **SDP-SWAP $= 1 - F$ and Bures$^2 = 2(1 - \sqrt{F})$ are two monotone functions of one fidelity.** They rank states identically — the same pairs are "closest" and "farthest" under both — but they are not the same distance. To see the ordering agreement without the ties of a short table, we sweep a continuous family of pure states and plot both against the fidelity.

In [ ]:
# A continuous tie-free family: |0> versus the qubit at polar angle theta, theta in (0, pi).
thetas = np.linspace(0.15, np.pi - 0.15, 12)
F_vals, swap_vals, bures_vals = [], [], []
for th in thetas:
    rho = density_matrix(KET_0)
    sigma = density_matrix(qubit_state(th))
    F = fidelity(rho, sigma)
    F_vals.append(F)
    swap_vals.append(quantum_wasserstein_squared_swap(rho, sigma))
    bures_vals.append(2.0 * (1.0 - np.sqrt(F)))

order = np.argsort(F_vals)
F_sorted = np.array(F_vals)[order]
swap_sorted = np.array(swap_vals)[order]
bures_sorted = np.array(bures_vals)[order]

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(F_sorted, swap_sorted, marker="o", color=COLORS["quantum"], linewidth=2.0,
        markersize=7, label=r"SDP-SWAP  $=1-F$  (coupling SDP, 04/04)")
ax.plot(F_sorted, bures_sorted, marker="s", color=COLORS["highlight"], linewidth=2.0,
        markersize=7, label=r"Bures$^2 = 2(1-\sqrt{F})$  (Bures bridge, 03/13)")
ax.set_xlabel(r"fidelity  $F = |\langle\psi|\phi\rangle|^2$")
ax.set_ylabel("transport-like cost on pure states")
ax.set_title("Two monotone functions of one fidelity: order-equivalent, not equal", pad=12)
ax.invert_xaxis()  # left-to-right: states grow farther apart (F: 1 -> 0)
ax.legend()
plt.show()

**Read the figure.** The horizontal axis is the fidelity $F$, **inverted** so that reading left to right the two states grow farther apart ($F$ falling from $1$ toward $0$). Two curves rise together: the sky-blue line is the SDP-SWAP cost $1 - F$ (a straight line in $F$), the rose line is the squared Bures distance $2(1 - \sqrt{F})$ (gently concave). The two curves meet only at $F = 1$ (both zero — identical states); as $F \to 0$ they separate — the SDP-SWAP rises to $1$, the squared Bures distance to $2$ — and the rose curve bows *above* the blue one through the whole interior, the gap widest near the middle. That picture is the claim in one glance: the curves never cross, so the two costs order every pair of states identically (order-equivalent); yet the rose curve is visibly not the blue line, so they are not equal. Reading this figure correctly is the difference between a true statement and an overclaim — Bures and the SDP-SWAP are relatives through the fidelity, not the same object.

## 5. The regimes table

Now the whole reconciliation on one page. The three objects relate *differently in different regimes* — and the discipline is to say which relationship holds where. Read each row as a regime; each cell says how the objects stand to one another there.

| Regime | Coupling SDP vs entropic | Coupling SDP vs Bures | What holds |
|---|---|---|---|
| **Commuting / diagonal** (quadratic cost) | entropic $\to$ SDP as $\varepsilon \to 0$; SDP $= W_2^2$ exactly | — (Bures uses the SWAP-cost / fidelity reading) | all three descend to the **same classical $W_2^2$ floor** (§2, §3) |
| **Pure states** (SWAP cost) | entropic $\to$ SDP as $\varepsilon \to 0$ | SDP-SWAP $= 1-F$, Bures$^2 = 2(1-\sqrt{F})$ | **order-equivalent, not equal**: both monotone in the same $F$ (§4) |
| **General mixed, non-commuting** | entropic $\to$ SDP as $\varepsilon \to 0$ (always) | no closed-form identity; cost-operator dependent | the entropic$\to$SDP limit is **universal**; SDP and Bures **genuinely differ** |

Read down the right-hand column and the family snaps into focus. The **entropic-to-SDP limit holds in every regime** — it is an algebraic property of the entropy term, not a coincidence of any special state, which is why the regression test pins it on the commuting pair and you can trust it everywhere. The **SDP-equals-classical-$W_2^2$** identity is special to the commuting/diagonal world, where coherence has nowhere to hide. And the **Bures-versus-SDP** relationship is *order-equivalence through the fidelity* on pure states, decaying to no fixed relationship at all once states are mixed and non-commuting and the cost operator matters. Three objects, one subject — equal where the structure forces it, order-equivalent where they share a fidelity, and honestly distinct otherwise. That nuance *is* the unified picture.

## Your turn

**Warm-up.** Extend the $\varepsilon$-sweep of §2 at the *large* end — add a couple of values above $\varepsilon = 2$ (say $\varepsilon = 4$ and $\varepsilon = 8$) on the same commuting diagonal pair, and watch the entropic transport cost climb further above the SDP value. Toward which simple coupling does the entropic plan tend as $\varepsilon$ grows without bound, and what does that say the transport cost should approach at the *high*-$\varepsilon$ end (the opposite limit from the one the regression test pins)?

**Go further.** Build a third reconciliation table of your own for the **diagonal collapse**, but with a *different* ground geometry: keep the commuting diagonal pair, change the positions from $\{0, 1, 2\}$ to non-uniform ones such as $\{0, 1, 4\}$, rebuild the quadratic-position cost, and check that the coupling SDP still equals `ot.emd2` on the diagonals with the matching classical cost $C_{ij} = (\text{pos}_i - \text{pos}_j)^2$. Does the diagonal-collapse identity care about *which* positions you choose, or only that the states commute? Explain what feature of the states — not the cost — is doing the work.

**Challenge.** Probe the boundary of the order-equivalence claim of §4. The statement "SDP-SWAP and Bures are order-equivalent" was made for **pure** states. Take a one-parameter family of *mixed* states — for instance $\rho_t = t\,|+\rangle\langle+| + (1 - t)\,I/2$ for several $t \in [0, 1]$ against a fixed reference — and compute, for each, both the squared Bures distance (via `bures_distance`) and the SDP-SWAP value (via `quantum_wasserstein_squared_swap`). Do the two still rank the states in the same order? Argue, from the fact that the clean $1 - F$ closed form was *derived for pure marginals*, why order-equivalence is a statement you should be cautious about extending off the pure-state case — and what you would need to check before claiming it more broadly.

## What you built

- You named the **three QOT objects** and their home notebooks — the **Bures bridge** (`03/13`), the **coupling SDP** (`04/04`), and the **entropic object** (`04/06`–`04/08`) — and put them on one bench.
- You watched the **entropic object converge to the coupling SDP** as $\varepsilon \to 0$, the gap collapsing onto the solver floor along a clean log–log line — the convergence the suite's regression test guards, now something you have *seen*.
- You confirmed the **diagonal-collapse common limit**: on commuting states the coupling SDP equals the classical $W_2^2$ to solver tolerance, the shared classical floor beneath all three objects.
- You stated the **Bures-versus-SDP relationship precisely** — SDP-SWAP $= 1 - F$ and Bures$^2 = 2(1 - \sqrt{F})$ are two monotone functions of one fidelity, hence **order-equivalent but not equal** — and read a **regimes table** that says exactly where the three coincide, where they only share an ordering, and where they genuinely differ.

Hold the whole picture for a moment, because you have earned it. Three times over you built a way to measure transport between quantum states, and you now know precisely how those three measurements relate: equal where the structure compels it, ordered alike where they share a fidelity, and distinct — honestly, interestingly distinct — otherwise. Refusing to flatten that into "they are all the same" is not a hedge; it is the accurate map, and accuracy is the whole reason the map is worth having.

## What's next

The reconciliation is complete, and the machinery is ready to point at real data — but a question stands between here and the capstone: a signal is not a quantum state, so *how do we turn one into the other?* In `10_embedding_signals_into_states` you will meet the menu of embeddings — the modelling choice that decides what structure of a signal the quantum geometry can even see.

## References

- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091
- G. De Palma & D. Trevisan, "Quantum optimal transport with quantum channels", *Annales Henri Poincaré* **22**, 3199–3234 (2021). DOI:10.1007/s00023-021-01042-3
- S. Cole, M. Lostaglio, K. Verma & M. M. Wilde, "Quantum Wasserstein distance based on an optimization over separable states", *IEEE Transactions on Information Theory* **69**, 6657–6678 (2023). DOI:10.1109/TIT.2023.3287993
- R. Bhatia, T. Jain & Y. Lim, "On the Bures–Wasserstein distance between positive definite matrices", *Expositiones Mathematicae* **37**, 165–191 (2019). DOI:10.1016/j.exmath.2018.01.002
- G. Peyré, L. Chizat, F.-X. Vialard & B. Schmitzer, "Quantum entropic regularization of matrix-valued optimal transport", *European Journal of Applied Mathematics* **30**, 1079–1102 (2019). DOI:10.1017/S0956792519000299

**Previous:** `notebooks/04_QuantumOptimalTransport/08_quantum_amari_bridge.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/10_embedding_signals_into_states.ipynb`